# Lecture 05: Supervised Learning and Decision Trees — Exercises

Companion to the note [Noter små](../Noter%20små.md).

Practise the supervised-learning vocabulary of Lecture 05 (losses, mean error, entropy, information gain, gain ratio), build decision trees by hand the way the exam asks, and then implement entropy, ID3, linear regression by gradient descent and a squashed linear (logistic) classifier from scratch.

**19 exercises** · 🧠 Concept ×3 · ✍️ By hand ×8 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import pandas as pd
from scipy.stats import entropy as sp_entropy
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(5)
# Hiking data set: is a hike enjoyable?
hikes = pd.DataFrame([
    ['sunny',  'short', 'friends', 'yes'],
    ['sunny',  'long',  'alone',   'no'],
    ['cloudy', 'short', 'friends', 'yes'],
    ['rainy',  'short', 'alone',   'no'],
    ['rainy',  'long',  'friends', 'no'],
    ['sunny',  'short', 'alone',   'yes'],
    ['cloudy', 'long',  'friends', 'yes'],
    ['sunny',  'long',  'alone',   'no'],
], columns=['Weather', 'Distance', 'Company', 'Enjoy'])
# Small regression data set (columns x1, x2) and targets
X_reg = np.array([[1., 2.], [0., 1.], [2., 0.], [1., 1.]])
y_reg = np.array([3., 1., 4., 2.])

## Part A · Concepts

### Exercise 1 · What kind of learning task?
*🧠 Concept · ★☆☆*

For each task say whether it is **classification** or **regression**, and whether learning is naturally **offline** or **online**:

1. Predict tomorrow's electricity price (in DKK/kWh) from today's prices, retrained once a year.
2. A spam filter that updates itself every time the user clicks "not spam".
3. Predict whether a running route is *fun* (yes/no) from length, nature and stoplights.
4. A robot vacuum learning the floor plan of a new home while cleaning.

Then explain in one sentence each: *interpolation vs extrapolation*, and *active learning*.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Regression = real-valued target, classification = a fixed finite set of values. Offline = all examples are available before acting.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Regression, offline. 2. Classification, online. 3. Classification, offline. 4. The target (map/occupancy of a cell) is categorical, and
   learning is online because examples arrive while acting.

*Interpolation* predicts for inputs between seen examples; *extrapolation* predicts beyond the range of the training data and is usually
much less accurate. *Active learning* is online learning in which the agent chooses which examples to obtain labels for (e.g. asks the user
about the emails it is least sure about).

</details>

### Exercise 2 · Success means generalisation
*🧠 Concept · ★☆☆*

A learner memorises every training example and answers "no" for unseen inputs. It has 0 training error.

1. Why is this not a successful learner according to the note?
2. What is *inductive bias*, and why is some bias unavoidable?
3. Give the inductive bias of a decision-tree learner that greedily picks the attribute with the highest information gain.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Without any preference among hypotheses that agree on the training data, nothing can be said about unseen examples.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Success is measured on **new** data. Memorisation says nothing about unseen inputs, so its test error can be terrible.
2. Inductive bias is the learner's preference for some hypotheses over others beyond consistency with the data. Many hypotheses fit any
   finite data set; without a preference there is no reason to predict one label rather than another on an unseen input.
3. A preference for **small (shallow) trees** that put highly informative attributes near the root (a form of Ockham's razor), plus the
   restriction to axis-aligned tests on single attributes.

</details>

### Exercise 3 · Why information gain loves many-valued attributes
*🧠 Concept · ★★☆*

Suppose we add an attribute *Date* to the hiking data, with a different value for each of the 8 hikes.

1. What is the expected entropy $\text{Entropy}(\text{Enjoy}\mid\text{Date})$ and hence the information gain of Date?
2. Why would a tree with Date at the root be useless?
3. How does the gain ratio fix this?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Each Date value has exactly one example. What is the entropy of a single-example subset?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Every branch contains one example, so every $h_i = 0$ and the expected entropy is 0. The gain equals the full entropy of Enjoy (1 bit here),
   the maximum possible.
2. Each leaf covers one training example, and a new hike has an unseen date, so the tree cannot classify it at all. It memorises instead of generalising.
3. The gain ratio divides by $\text{SplitInformation} = -\sum_i \frac{|S_i|}{|S|}\log_2\frac{|S_i|}{|S|}$, which is large ($\log_2 8 = 3$) for
   attributes that shatter the data into many small subsets. See the by-hand exercise below for the numbers.

</details>

## Part B · By hand

### Exercise 4 · Mean squared error of two linear models
*✍️ By hand · ★☆☆*

For the data

| $x_1$ | $x_2$ | $y$ |
|:--:|:--:|:--:|
| 1 | 2 | 3 |
| 0 | 1 | 1 |
| 2 | 0 | 4 |
| 1 | 1 | 2 |

compute $\hat y$ and the MSE of $\hat Y_w = w_0 + w_1x_1 + w_2x_2$ for (i) $w = (0, 1, 1)$ and (ii) $w = (1, 1, 0)$. Which model is preferable?

In [ ]:
mse_i = None
mse_ii = None

check('MSE (i)', mse_i, np.mean((X_reg @ [1, 1] - y_reg) ** 2))
check('MSE (ii)', mse_ii, np.mean((1 + X_reg @ [1, 0] - y_reg) ** 2))

<details>
<summary><b>💡 Hint</b></summary>

(i) gives $\hat y = x_1 + x_2$; (ii) gives $\hat y = 1 + x_1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(i) $\hat y = (3, 1, 2, 2)$, errors $(0, 0, -2, 0)$, SSE $= 4$, **MSE $= 1$**.
(ii) $\hat y = (2, 1, 3, 2)$, errors $(-1, 0, -1, 0)$, SSE $= 2$, **MSE $= 0.5$**. Model (ii) is preferable.

```python
mse_i = 1.0
mse_ii = 0.5
```

</details>

### Exercise 5 · Four losses on the same predictions
*✍️ By hand · ★☆☆*

Predictions $p = (2, 3, 5, 4)$ and actual values $a = (2, 4, 2, 4.5)$. Compute the mean 0–1 loss, mean absolute loss, mean squared loss
and the worst-case ($L_\infty$) loss. Which loss is most affected by the third example?

In [ ]:
l0 = None
l1 = None
l2 = None
linf = None

_p = np.array([2, 3, 5, 4.]); _a = np.array([2, 4, 2, 4.5])
check('0-1', l0, np.mean(_p != _a)); check('L1', l1, np.mean(np.abs(_p - _a)))
check('L2', l2, np.mean((_p - _a) ** 2)); check('Linf', linf, np.max(np.abs(_p - _a)))

<details>
<summary><b>💡 Hint</b></summary>

Errors $p - a = (0, -1, 3, -0.5)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

0–1: three wrong out of four, $0.75$. $L_1$: $(0+1+3+0.5)/4 = 1.125$. $L_2$: $(0+1+9+0.25)/4 = 2.5625$. $L_\infty$: $3$.
The squared loss is dominated by the third example (9 of the total 10.25), and $L_\infty$ only looks at it.

```python
l0 = 0.75
l1 = 1.125
l2 = 2.5625
linf = 3.0
```

</details>

### Exercise 6 · Entropy of a fair and a loaded die
*✍️ By hand · ★☆☆*

Compute (in bits) the entropy of a fair six-sided die and of a loaded die with $P(6) = 0.4$, $P(5) = 0.2$ and the four other outcomes
equally likely. Explain the difference.

In [ ]:
H_fair = None
H_loaded = None

check('fair', H_fair, sp_entropy(np.ones(6) / 6, base=2))
check('loaded', H_loaded, sp_entropy([0.1, 0.1, 0.1, 0.1, 0.2, 0.4], base=2))

<details>
<summary><b>💡 Hint</b></summary>

$H = -\sum_i p_i\log_2 p_i$. For the loaded die, the other four outcomes share $0.4$, so each has $0.1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Fair: $\log_2 6 \approx 2.585$ bits. Loaded: $-(0.4\log_2 0.4 + 0.2\log_2 0.2 + 4\cdot 0.1\log_2 0.1) = 0.529 + 0.464 + 1.329 \approx 2.322$ bits.
The uniform distribution has maximal entropy; the loaded die is more predictable (purer), so its entropy is lower.

```python
H_fair = np.log2(6)
H_loaded = -(0.4 * np.log2(0.4) + 0.2 * np.log2(0.2) + 4 * 0.1 * np.log2(0.1))
```

</details>

### Exercise 7 · Information gain and the root split
*✍️ By hand · ★★☆*

For the hiking data (shown below) compute $\text{Entropy}(\text{Enjoy})$ and the information gain of Weather, Distance and Company.
Which attribute becomes the root?

| # | Weather | Distance | Company | Enjoy |
|:--:|:--:|:--:|:--:|:--:|
| 1 | sunny | short | friends | yes |
| 2 | sunny | long | alone | no |
| 3 | cloudy | short | friends | yes |
| 4 | rainy | short | alone | no |
| 5 | rainy | long | friends | no |
| 6 | sunny | short | alone | yes |
| 7 | cloudy | long | friends | yes |
| 8 | sunny | long | alone | no |

In [ ]:
H_enjoy = None
ig = None   # dict, e.g. {'Weather': ..., 'Distance': ..., 'Company': ...}
root = None # 'Weather', 'Distance' or 'Company'

_ig = {c: mutual_info_score(hikes[c], hikes['Enjoy']) / np.log(2) for c in ['Weather', 'Distance', 'Company']}
check('Entropy(Enjoy)', H_enjoy, 1.0)
for _c in _ig:
    check(f'IG({_c})', None if ig is None else ig.get(_c), _ig[_c], tol=1e-3)
check('root', root, max(_ig, key=_ig.get))

<details>
<summary><b>💡 Hint 1</b></summary>

4 yes and 4 no, so $\text{Entropy}(\text{Enjoy}) = 1$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Weather: sunny has 2 yes / 2 no, cloudy 2 yes / 0 no, rainy 0 yes / 2 no. Distance: short 3/1, long 1/3. Company: friends 3/1, alone 1/3.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\text{Entropy}(\text{Enjoy}) = 1$.

* Weather: $\tfrac48\cdot1 + \tfrac28\cdot0 + \tfrac28\cdot0 = 0.5$, gain $= 0.5$.
* Distance: $\text{Entropy}(3/4,1/4) = 0.811$ in both branches, so expected entropy $0.811$ and gain $\approx 0.189$.
* Company: the same counts, gain $\approx 0.189$.

**Weather** is the root. (Information gain is the mutual information between attribute and class, which is what the check uses as a reference.)

```python
H_enjoy = 1.0
ig = {'Weather': 0.5, 'Distance': 1 - 0.8112781, 'Company': 1 - 0.8112781}
root = 'Weather'
```

</details>

### Exercise 8 · Finish the tree and classify
*✍️ By hand · ★★☆*

Continue ID3 below the root. Only the *sunny* branch is impure. Compute the information gain of Distance and Company inside that branch,
draw the final tree and use it to classify (a) cloudy/long/alone and (b) sunny/short/friends.

In [ ]:
ig_sunny_distance = None
ig_sunny_company = None
pred_a = None  # 'yes' or 'no'
pred_b = None

_s = hikes[hikes.Weather == 'sunny']
check('IG(Distance | sunny)', ig_sunny_distance, mutual_info_score(_s.Distance, _s.Enjoy) / np.log(2), tol=1e-3)
check('IG(Company | sunny)', ig_sunny_company, mutual_info_score(_s.Company, _s.Enjoy) / np.log(2), tol=1e-3)
check('prediction (a)', pred_a, 'yes'); check('prediction (b)', pred_b, 'yes')

<details>
<summary><b>💡 Hint</b></summary>

Sunny rows: 1 (short, friends, yes), 2 (long, alone, no), 6 (short, alone, yes), 8 (long, alone, no).

</details>

<details>
<summary><b>✅ Solution</b></summary>

In the sunny branch (2 yes, 2 no; entropy 1): Distance separates perfectly (short → yes, long → no), gain $1$. Company: friends → 1 yes;
alone → 1 yes, 2 no (entropy 0.918), expected entropy $\tfrac34\cdot0.918 = 0.689$, gain $\approx 0.311$. Choose **Distance**.

```
Weather = cloudy → yes
Weather = rainy  → no
Weather = sunny  → Distance = short → yes
                   Distance = long  → no
```
(a) cloudy → **yes**. (b) sunny, short → **yes**. Company never appears in the tree.

```python
ig_sunny_distance = 1.0
ig_sunny_company = 1 - 0.75 * 0.9182958
pred_a = 'yes'
pred_b = 'yes'
```

</details>

### Exercise 9 · Gain ratio versus information gain
*✍️ By hand · ★★☆*

Add the attribute *Date* with eight distinct values to the hiking data. Compute the split information and the gain ratio of Date,
Weather and Distance.

In [ ]:
gr_date = None
gr_weather = None
gr_distance = None

_H = lambda col: sp_entropy(hikes[col].value_counts(), base=2)
check('GR(Date)', gr_date, 1.0 / np.log2(8), tol=1e-3)
check('GR(Weather)', gr_weather, (mutual_info_score(hikes.Weather, hikes.Enjoy) / np.log(2)) / _H('Weather'), tol=1e-3)
check('GR(Distance)', gr_distance, (mutual_info_score(hikes.Distance, hikes.Enjoy) / np.log(2)) / _H('Distance'), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

SplitInformation is the entropy of the attribute's own value distribution: Weather has 4/2/2 examples per value.

</details>

<details>
<summary><b>✅ Solution</b></summary>

* Date: gain $1$, split information $\log_2 8 = 3$, gain ratio $1/3$.
* Weather: gain $0.5$, split information $-(\tfrac12\log_2\tfrac12 + 2\cdot\tfrac14\log_2\tfrac14) = 1.5$, gain ratio $1/3$.
* Distance: gain $0.189$, split information $1$, gain ratio $0.189$.

The gain ratio cuts Date from the clear winner (1.0 vs 0.5) down to a tie with Weather. In practice C4.5 additionally only considers
attributes with at least average information gain, and Date-like identifiers are usually removed beforehand.

```python
gr_date = 1 / 3
gr_weather = 0.5 / 1.5
gr_distance = 1 - 0.8112781
```

</details>

### Exercise 10 · Sigmoid, its derivative, softmax and log loss
*✍️ By hand · ★★☆*

1. Compute $\text{sigmoid}(2)$ and its derivative at $2$ using $\sigma'(x) = \sigma(x)(1-\sigma(x))$. What is the largest possible value of $\sigma'$?
2. Compute $\text{softmax}(1, 2, 3)$.
3. Compute the mean binary log loss (natural log) of predictions $p = (0.9, 0.6, 0.2, 0.99)$ for actual values $a = (1, 0, 0, 1)$.
   Which example contributes most?

In [ ]:
sig2 = None
dsig2 = None
max_dsig = None
sm = None       # list of 3
mean_logloss = None

from scipy.special import expit, softmax as _softmax
check('sigmoid(2)', sig2, expit(2), tol=1e-3); check("sigmoid'(2)", dsig2, expit(2) * (1 - expit(2)), tol=1e-3)
check("max sigmoid'", max_dsig, 0.25)
check('softmax', sm, _softmax([1, 2, 3]), tol=1e-3)
_p = np.array([0.9, 0.6, 0.2, 0.99]); _a = np.array([1, 0, 0, 1])
check('log loss', mean_logloss, np.mean(-_a * np.log(_p) - (1 - _a) * np.log(1 - _p)), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$e^{-2} \approx 0.1353$. For softmax, $e^1, e^2, e^3 \approx 2.718, 7.389, 20.086$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\sigma(2) = 1/(1+0.1353) \approx 0.8808$, $\sigma'(2) = 0.8808\cdot0.1192 \approx 0.1050$. The maximum is $\sigma'(0) = 0.25$.
2. Sum $= 30.193$, so softmax $\approx (0.090, 0.245, 0.665)$.
3. Losses: $-\ln 0.9 = 0.105$, $-\ln 0.4 = 0.916$, $-\ln 0.8 = 0.223$, $-\ln 0.99 = 0.010$; mean $\approx 0.314$.
   The second example (predicted 0.6 for a 0) dominates; log loss punishes confident wrong predictions hardest.

```python
sig2 = 1 / (1 + np.exp(-2))
dsig2 = sig2 * (1 - sig2)
max_dsig = 0.25
sm = list(np.exp([1, 2, 3]) / np.exp([1, 2, 3]).sum())
mean_logloss = np.mean([-np.log(0.9), -np.log(0.4), -np.log(0.8), -np.log(0.99)])
```

</details>

### Exercise 11 · Deriving the gradient of the mean squared loss
*✍️ By hand · ★★☆*

Starting from $\text{error}(E_s, w) = \frac{1}{|E_s|}\sum_{e}\big(\sum_i w_iX_i(e) - Y(e)\big)^2$, derive equation (7.2) of the note.
Then compute the gradient at $w = (0, 0, 0)$ for the regression data of the MSE exercise (with $X_0 = 1$).

In [ ]:
grad0 = None  # [d/dw0, d/dw1, d/dw2]

_Xb = np.c_[np.ones(4), X_reg]
check('gradient at 0', grad0, 2 / 4 * _Xb.T @ (_Xb @ np.zeros(3) - y_reg))

<details>
<summary><b>💡 Hint</b></summary>

Chain rule: $\frac{\partial}{\partial w_i}\delta(e)^2 = 2\delta(e)\frac{\partial\delta(e)}{\partial w_i} = 2\delta(e)X_i(e)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{\partial}{\partial w_i}\text{error} = \frac{1}{|E_s|}\sum_e 2\,\delta(e)\,X_i(e)$ with $\delta(e) = \hat Y_w(e) - Y(e)$, which is (7.2).

At $w = 0$, $\delta = -y = (-3, -1, -4, -2)$. Then
$\partial_{w_0} = \tfrac24(-10) = -5$, $\partial_{w_1} = \tfrac24(-3 + 0 - 8 - 2) = -6.5$, $\partial_{w_2} = \tfrac24(-6 - 1 + 0 - 2) = -4.5$.

```python
grad0 = [-5.0, -6.5, -4.5]
```

</details>

## Part C · Code

### Exercise 12 · Entropy of a label column
*💻 Code · ★☆☆*

Implement `entropy(labels)` in bits for any sequence of class labels (use `np.unique(..., return_counts=True)`), with the convention
$0\log 0 = 0$. Test it on `hikes['Enjoy']` and on a pure column.

In [ ]:
def entropy(labels):
    # TODO
    return None

H_hikes = entropy(hikes['Enjoy'])
H_pure = entropy(['yes'] * 5)

check('entropy(Enjoy)', H_hikes, sp_entropy(hikes.Enjoy.value_counts(), base=2))
check('pure column', H_pure, 0.0)

<details>
<summary><b>💡 Hint</b></summary>

Counts with `np.unique` never contain zeros, so $0\log 0$ never actually occurs.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A pure column has entropy 0 (the note's convention $\text{logloss}(1,1) = 0$); a 50/50 column has entropy 1 bit.

```python
def entropy(labels):
    _, counts = np.unique(np.asarray(labels), return_counts=True)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())

H_hikes = entropy(hikes['Enjoy'])
H_pure = entropy(['yes'] * 5)
```

</details>

### Exercise 13 · Information gain and gain ratio
*💻 Code · ★★☆*

Implement `info_gain(df, attr, target)` and `gain_ratio(df, attr, target)` using your `entropy`. Print both for every attribute of `hikes`.

In [ ]:
def info_gain(df, attr, target):
    # TODO: Entropy(target) - sum_i q_i * h_i
    return None

def gain_ratio(df, attr, target):
    # TODO
    return None

ig_weather = info_gain(hikes, 'Weather', 'Enjoy')
gr_company = gain_ratio(hikes, 'Company', 'Enjoy')

check('IG(Weather)', ig_weather, mutual_info_score(hikes.Weather, hikes.Enjoy) / np.log(2))
check('GR(Company)', gr_company, mutual_info_score(hikes.Company, hikes.Enjoy) / np.log(2) / sp_entropy(hikes.Company.value_counts(), base=2))

<details>
<summary><b>💡 Hint</b></summary>

`for v, sub in df.groupby(attr): ...` gives each subset $E_i$; $q_i = $ `len(sub) / len(df)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The split information is simply `entropy(df[attr])`, the entropy of the attribute's own values.

```python
def info_gain(df, attr, target):
    exp_ent = sum(len(sub) / len(df) * entropy(sub[target]) for _, sub in df.groupby(attr))
    return entropy(df[target]) - exp_ent

def gain_ratio(df, attr, target):
    return info_gain(df, attr, target) / entropy(df[attr])

ig_weather = info_gain(hikes, 'Weather', 'Enjoy')
gr_company = gain_ratio(hikes, 'Company', 'Enjoy')
for a in ['Weather', 'Distance', 'Company']:
    print(a, round(info_gain(hikes, a, 'Enjoy'), 3), round(gain_ratio(hikes, a, 'Enjoy'), 3))
```

</details>

### Exercise 14 · ID3 from scratch
*💻 Code · ★★★*

Implement the recursive learner `id3(df, attrs, target)` returning a nested dict `{attr: {value: subtree}}` with class labels at the leaves:

* if all examples have the same label → return that label,
* if no attributes remain → return the majority label,
* otherwise split on the attribute with the highest information gain and recurse on each subset (removing that attribute).

Also implement `predict(tree, row)` (`row` is a dict or a pandas Series). Learn the tree on `hikes`, print it and compute its training accuracy.

In [ ]:
def id3(df, attrs, target):
    # TODO
    return None

def predict(tree, row):
    # TODO: walk down the dict until you reach a label
    return None

tree = id3(hikes, ['Weather', 'Distance', 'Company'], 'Enjoy')
train_acc = None

check('root attribute', None if tree is None else list(tree)[0], 'Weather')
check('training accuracy', train_acc, 1.0)
check('predict cloudy/long/alone', None if tree is None else predict(tree, {'Weather': 'cloudy', 'Distance': 'long', 'Company': 'alone'}), 'yes')

<details>
<summary><b>💡 Hint 1</b></summary>

`max(attrs, key=lambda a: info_gain(df, a, target))` picks the best attribute.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

In `predict`: `while isinstance(tree, dict): attr = next(iter(tree)); tree = tree[attr][row[attr]]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The learned tree is `{'Weather': {'cloudy': 'yes', 'rainy': 'no', 'sunny': {'Distance': {'long': 'no', 'short': 'yes'}}}}`, matching
the by-hand exercise. Training accuracy is 1 because the data are consistent (no two identical inputs with different labels).
A real implementation would also need a default for attribute values never seen in training (e.g. the majority label at that node).

```python
def id3(df, attrs, target):
    labels = df[target]
    if labels.nunique() == 1:
        return labels.iloc[0]
    if not attrs:
        return labels.mode().iloc[0]
    best = max(attrs, key=lambda a: info_gain(df, a, target))
    rest = [a for a in attrs if a != best]
    return {best: {v: id3(sub, rest, target) for v, sub in df.groupby(best)}}

def predict(tree, row):
    while isinstance(tree, dict):
        attr = next(iter(tree))
        tree = tree[attr][row[attr]]
    return tree

tree = id3(hikes, ['Weather', 'Distance', 'Company'], 'Enjoy')
print(tree)
train_acc = np.mean([predict(tree, r) == r['Enjoy'] for _, r in hikes.iterrows()])
```

</details>

### Exercise 15 · Linear regression by gradient descent
*💻 Code · ★★☆*

Implement `fit_linear_gd(X, y, lr, epochs)` that adds the constant feature $X_0 = 1$, starts from $w = 0$ and repeatedly applies
$w_i \leftarrow w_i - \eta\,\frac{\partial}{\partial w_i}\text{error}$ using equation (7.2). Fit it on `X_reg, y_reg` and compare with the exact
least-squares solution from `np.linalg.lstsq`.

In [ ]:
def fit_linear_gd(X, y, lr=0.05, epochs=5000):
    # TODO
    return None

w_gd = fit_linear_gd(X_reg, y_reg)

_Xb = np.c_[np.ones(len(X_reg)), X_reg]
check('GD solution', w_gd, np.linalg.lstsq(_Xb, y_reg, rcond=None)[0], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Vectorised gradient: `2 / N * Xb.T @ (Xb @ w - y)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With 4 examples and 3 weights the least-squares solution is $w \approx (0.5, 1.667, 0.333)$, which GD reaches after a few thousand steps.
If you raise the learning rate too much (try `lr=0.5`), the updates overshoot and diverge.

```python
def fit_linear_gd(X, y, lr=0.05, epochs=5000):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        w -= lr * 2 / len(y) * Xb.T @ (Xb @ w - y)
    return w

w_gd = fit_linear_gd(X_reg, y_reg)
print(w_gd)
```

</details>

### Exercise 16 · A squashed linear classifier (logistic regression)
*💻 Code · ★★☆*

Implement `fit_logistic(X, y, lr, epochs)`: $\hat Y_w(e) = \text{sigmoid}(\sum_i w_iX_i(e))$ trained by gradient descent on the **mean log loss**,
whose gradient is $\frac1N X^\top(\hat y - y)$. Train on the standardised data below and compare the predicted probabilities with
`LogisticRegression(C=np.inf)` (no regularisation).

In [ ]:
X_clf, y_clf = make_classification(n_samples=200, n_features=2, n_redundant=0, n_informative=2,
                                   class_sep=0.8, flip_y=0.1, random_state=1)
X_clf = (X_clf - X_clf.mean(0)) / X_clf.std(0)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def fit_logistic(X, y, lr=0.5, epochs=5000):
    # TODO: return w = [w0, w1, w2]
    return None

w_log = fit_logistic(X_clf, y_clf)
proba_mine = None if w_log is None else sigmoid(np.c_[np.ones(len(X_clf)), X_clf] @ w_log)

check('probabilities match sklearn', proba_mine, LogisticRegression(C=np.inf, tol=1e-10, max_iter=10000).fit(X_clf, y_clf).predict_proba(X_clf)[:, 1], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

The derivative of the log loss through the sigmoid simplifies beautifully: $\partial/\partial w_i = (\hat y - y)X_i$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Combining $\text{logloss}$ with $\sigma'(x) = \sigma(x)(1-\sigma(x))$ cancels the $\sigma(1-\sigma)$ factor, leaving the same
"error times input" form as linear regression. The log loss is convex in $w$, so GD finds the same optimum as sklearn.

```python
X_clf, y_clf = make_classification(n_samples=200, n_features=2, n_redundant=0, n_informative=2,
                                   class_sep=0.8, flip_y=0.1, random_state=1)
X_clf = (X_clf - X_clf.mean(0)) / X_clf.std(0)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def fit_logistic(X, y, lr=0.5, epochs=5000):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        w -= lr * Xb.T @ (sigmoid(Xb @ w) - y) / len(y)
    return w

w_log = fit_logistic(X_clf, y_clf)
proba_mine = sigmoid(np.c_[np.ones(len(X_clf)), X_clf] @ w_log)
```

</details>

### Exercise 17 · A numerically stable softmax
*💻 Code · ★★☆*

`np.exp(1000)` overflows. Implement `softmax(alpha)` for a 1-D array that works for inputs like `[1000, 1001, 1002]`, using the fact that
softmax is unchanged when the same constant is subtracted from every component. Prove that fact in one line.

In [ ]:
def softmax(alpha):
    # TODO
    return None

s_big = softmax(np.array([1000., 1001., 1002.]))

from scipy.special import softmax as _sm
check('stable softmax', s_big, _sm([1000., 1001., 1002.]))

<details>
<summary><b>💡 Hint</b></summary>

Subtract `alpha.max()` first.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{e^{\alpha_i - c}}{\sum_j e^{\alpha_j - c}} = \frac{e^{-c}e^{\alpha_i}}{e^{-c}\sum_j e^{\alpha_j}}$, so the shift cancels. With $c = \max_j\alpha_j$ all
exponents are $\le 0$ and nothing overflows. The result equals $\text{softmax}(0, 1, 2) \approx (0.090, 0.245, 0.665)$.

```python
def softmax(alpha):
    z = np.exp(alpha - np.max(alpha))
    return z / z.sum()

s_big = softmax(np.array([1000., 1001., 1002.]))
```

</details>

### Exercise 18 · Tree depth, training error and test error
*💻 Code · ★★☆*

Generate `make_moons(n_samples=400, noise=0.3, random_state=0)`, split 50/50 (`random_state=0`), and fit
`DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=0)` for $d = 1,\dots,12$. Store training and test accuracies in
lists `train_accs`, `test_accs`, and the depth with the best test accuracy in `best_depth`. What happens to the gap between the two curves?

In [ ]:
train_accs = None
test_accs = None
best_depth = None

if train_accs is not None:
    check('training accuracy reaches 1 for deep trees', train_accs[-1], 1.0)
    check('best depth maximises test accuracy', test_accs[best_depth - 1], max(test_accs))
    check('deepest tree is not the best on test data', test_accs[-1] < max(test_accs), True)
else:
    check('training accuracy reaches 1 for deep trees', None, 1.0)

<details>
<summary><b>💡 Hint</b></summary>

Use `train_test_split(X, y, test_size=0.5, random_state=0)` and `.score(...)` for accuracy.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Training accuracy rises monotonically to 1.0, but test accuracy peaks at a moderate depth and then drops: deep trees fit the label noise.
This is overfitting (Lecture 07), and it is why trees are pruned or depth-limited using a validation set.

```python
X_m, y_m = make_moons(n_samples=400, noise=0.3, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X_m, y_m, test_size=0.5, random_state=0)
train_accs, test_accs = [], []
for d in range(1, 13):
    t = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=0).fit(Xtr, ytr)
    train_accs.append(t.score(Xtr, ytr)); test_accs.append(t.score(Xte, yte))
best_depth = int(np.argmax(test_accs)) + 1
print(np.round(train_accs, 3)); print(np.round(test_accs, 3)); print('best depth', best_depth)
```

</details>

### Exercise 19 · The curse of dimensionality
*💻 Code · ★★★*

For $d \in \{1, 2, 10, 100, 1000\}$ draw 500 points uniformly in $[0,1]^d$ and one query point. Compute the ratio
$\frac{\text{distance to nearest point}}{\text{distance to farthest point}}$ and store the ratios in a list `ratios`.
What does this mean for learners that rely on "nearby" training examples?

In [ ]:
rng_c = np.random.default_rng(0)
ratios = None

check('ratio grows towards 1 with d', None if ratios is None else bool(ratios[-1] > 0.8 and ratios[0] < 0.05 and np.all(np.diff(ratios) > 0)), True)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.norm(X - q, axis=1)` gives all distances at once.

</details>

<details>
<summary><b>✅ Solution</b></summary>

In 1-D the nearest point is almost on top of the query (ratio ≈ 0.00x), but in 1000-D the nearest and farthest points are nearly equally
far away (ratio ≈ 0.9). Data become sparse: to cover the space you need exponentially many examples, and "nearest neighbours" stop
being meaningfully near. This is the curse of dimensionality from the note.

```python
rng_c = np.random.default_rng(0)
ratios = []
for d in (1, 2, 10, 100, 1000):
    X = rng_c.random((500, d)); q = rng_c.random(d)
    dist = np.linalg.norm(X - q, axis=1)
    ratios.append(dist.min() / dist.max())
print(np.round(ratios, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Noter små](../Noter%20små.md).*